# 📊 TRABALHO FINAL: ANÁLISE ESTATÍSTICA COMPLETA
## Simulação: TCL, Intervalo de Confiança e ANOVA

**Data**: 27 de dezembro de 2025  
**Disciplina**: Planejamento e Análise Estatística de Experimentos  
**Referência**: Estudos Dirigidos 1 e 2 - PDF extraídos

### Estrutura do Trabalho
- **Parte 1**: Teorema Central do Limite (TCL) com distribuição exponencial
- **Parte 2**: Cobertura do Intervalo de Confiança (IC)
- **Parte 3**: ANOVA One-Way com teste Post-Hoc de Tukey HSD

---

In [2]:
# IMPORTAÇÕES CONSOLIDADAS
# ====================================================
import os
import json
from datetime import datetime
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo dos gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 10

# CONFIGURAÇÃO: Diretórios de Saída Organizados por Método
# ====================================================
BASE_RESULTADO = r'D:\GitHub\DoutoradoCefet\PlanejamentoAnaliseEstatisticoExperimentos\Trabalhos\estudo_dirigido_3\relatorio\resultados'
BASE_DADOS = os.path.join(BASE_RESULTADO, 'dados')
BASE_IMAGENS = os.path.join(BASE_RESULTADO, 'imagens')

# Pastas por método
METODOS = {
    'tcl': {
        'dados': os.path.join(BASE_DADOS, 'tcl'),
        'imagens': os.path.join(BASE_IMAGENS, 'tcl')
    },
    'ic': {
        'dados': os.path.join(BASE_DADOS, 'ic'),
        'imagens': os.path.join(BASE_IMAGENS, 'ic')
    },
    'anova': {
        'dados': os.path.join(BASE_DADOS, 'anova'),
        'imagens': os.path.join(BASE_IMAGENS, 'anova')
    }
}

# Criar diretórios se não existirem
for metodo, pastas in METODOS.items():
    for pasta in pastas.values():
        os.makedirs(pasta, exist_ok=True)

# Função para salvar dados por método
def salvar_dados(dados, nome_arquivo, metodo='tcl', tipo='csv'):
    """Salvar dados em CSV ou JSON no diretório do método"""
    pasta = METODOS[metodo]['dados']
    caminho = os.path.join(pasta, f"{nome_arquivo}.{tipo}")
    if tipo == 'csv':
        if isinstance(dados, pd.DataFrame):
            dados.to_csv(caminho, index=False)
        else:
            pd.DataFrame(dados).to_csv(caminho, index=False)
    elif tipo == 'json':
        with open(caminho, 'w') as f:
            json.dump(dados, f, indent=2)
    print(f"✅ Salvo: {caminho}")

# Função para salvar figuras por método
def salvar_figura(fig, nome_arquivo, metodo='tcl'):
    """Salvar figura em PNG no diretório do método"""
    pasta = METODOS[metodo]['imagens']
    caminho = os.path.join(pasta, f"{nome_arquivo}.png")
    fig.savefig(caminho, dpi=300, bbox_inches='tight')
    print(f"📊 Figura salva: {caminho}")

# Dicionário de parâmetros
PARAMETROS = {
    'data_execucao': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'disciplina': 'Planejamento e Análise Estatística de Experimentos',
    'referencia': 'Estudos Dirigidos 1 e 2',
    'estrutura': 'Organizada por método (TCL, IC, ANOVA)'
}

print("✅ Importações e configuração inicializadas!")
print(f"\n📁 ESTRUTURA DE DIRETÓRIOS:")
print(f"   TCL:")
print(f"      Dados: {METODOS['tcl']['dados']}")
print(f"      Imagens: {METODOS['tcl']['imagens']}")
print(f"   IC:")
print(f"      Dados: {METODOS['ic']['dados']}")
print(f"      Imagens: {METODOS['ic']['imagens']}")
print(f"   ANOVA:")
print(f"      Dados: {METODOS['anova']['dados']}")
print(f"      Imagens: {METODOS['anova']['imagens']}")

✅ Importações e configuração inicializadas!

📁 ESTRUTURA DE DIRETÓRIOS:
   TCL:
      Dados: D:\GitHub\DoutoradoCefet\PlanejamentoAnaliseEstatisticoExperimentos\Trabalhos\estudo_dirigido_3\relatorio\resultados\dados\tcl
      Imagens: D:\GitHub\DoutoradoCefet\PlanejamentoAnaliseEstatisticoExperimentos\Trabalhos\estudo_dirigido_3\relatorio\resultados\imagens\tcl
   IC:
      Dados: D:\GitHub\DoutoradoCefet\PlanejamentoAnaliseEstatisticoExperimentos\Trabalhos\estudo_dirigido_3\relatorio\resultados\dados\ic
      Imagens: D:\GitHub\DoutoradoCefet\PlanejamentoAnaliseEstatisticoExperimentos\Trabalhos\estudo_dirigido_3\relatorio\resultados\imagens\ic
   ANOVA:
      Dados: D:\GitHub\DoutoradoCefet\PlanejamentoAnaliseEstatisticoExperimentos\Trabalhos\estudo_dirigido_3\relatorio\resultados\dados\anova
      Imagens: D:\GitHub\DoutoradoCefet\PlanejamentoAnaliseEstatisticoExperimentos\Trabalhos\estudo_dirigido_3\relatorio\resultados\imagens\anova


# 📈 PARTE 1: Teorema Central do Limite (TCL)

## Objetivo
Demonstrar numericamente que o TCL válido mesmo com distribuições assimétricas (Exponencial).

## Setup
- **Distribuição**: Exponencial com λ = 0.2
- **Média populacional**: μ = 1/λ = 5
- **Tamanhos amostrais**: n = 5 (pequena) e n = 50 (grande)
- **Número de simulações**: k = 10.000

## Expectativa
- A assimetria da população é forte (skew ≈ 2)
- A assimetria das médias amostrais reduz com aumento de n
- Para n = 50, a distribuição das médias deve ser aproximadamente Normal

---

In [ ]:
# PARTE 1: SIMULAÇÃO DO TCL
print("\n" + "="*70)
print("🎯 PARTE 1: TEOREMA CENTRAL DO LIMITE")
print("="*70)

# Parâmetros da Parte 1
TRUE_MEAN = 5.0
RATE = 0.2
POP_SIZE = 100000
NUM_SIMULATIONS = 10000
SAMPLE_SIZES = [5, 50]

PARAMETROS['parte_1'] = {
    'distribuicao': 'Exponencial',
    'lambda': RATE,
    'media_populacional': TRUE_MEAN,
    'tamanho_populacao': POP_SIZE,
    'num_simulacoes': NUM_SIMULATIONS,
    'tamanhos_amostrais': SAMPLE_SIZES
}

# Gerar população exponencial
np.random.seed(42)
population = np.random.exponential(scale=1/RATE, size=POP_SIZE)

# Análise da população
pop_mean = np.mean(population)
pop_std = np.std(population, ddof=0)
pop_skewness = stats.skew(population)
pop_kurtosis = stats.kurtosis(population)

print(f"\n📊 POPULAÇÃO EXPONENCIAL:")
print(f"   Média: {pop_mean:.4f} (esperado: {TRUE_MEAN})")
print(f"   Desvio padrão: {pop_std:.4f}")
print(f"   Assimetria: {pop_skewness:.4f} (forte assimetria à direita)")
print(f"   Curtose: {pop_kurtosis:.4f}")

# Simulação do TCL para n=5 e n=50
resultados_tcl = {}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, n in enumerate(SAMPLE_SIZES):
    print(f"\n🔍 TAMANHO AMOSTRAL: n = {n}")
    
    # Gerar médias amostrais
    means = np.array([np.mean(np.random.choice(population, size=n, replace=True)) 
                     for _ in range(NUM_SIMULATIONS)])
    resultados_tcl[n] = means
    
    # Estatísticas das médias
    mean_of_means = np.mean(means)
    std_of_means = np.std(means, ddof=1)
    teorico_std_of_means = pop_std / np.sqrt(n)
    skewness_means = stats.skew(means)
    
    print(f"   Média das médias: {mean_of_means:.4f}")
    print(f"   DP das médias (observado): {std_of_means:.4f}")
    print(f"   DP das médias (teórico): {teorico_std_of_means:.4f}")
    print(f"   Assimetria das médias: {skewness_means:.4f}")
    
    # Visualizar histograma
    ax = axes[idx]
    ax.hist(means, bins=50, color='skyblue', alpha=0.7, edgecolor='black', density=True)
    
    # Sobrepor curva normal teórica
    x = np.linspace(means.min(), means.max(), 100)
    y_normal = stats.norm.pdf(x, pop_mean, teorico_std_of_means)
    ax.plot(x, y_normal, 'r-', linewidth=2, label='Normal teórica')
    
    ax.axvline(mean_of_means, color='green', linestyle='--', linewidth=2, label=f'Média = {mean_of_means:.2f}')
    ax.set_xlabel('Média da Amostra')
    ax.set_ylabel('Densidade')
    ax.set_title(f'TCL: n = {n} (Assimetria = {skewness_means:.4f})')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Salvar figura do TCL (em pasta TCL)
salvar_figura(fig, '01_tcl_simulacao', metodo='tcl')

# Salvar resultados do TCL (em pasta TCL)
tcl_results = pd.DataFrame({
    'Tamanho_Amostra': SAMPLE_SIZES,
    'Média_das_Médias': [np.mean(resultados_tcl[n]) for n in SAMPLE_SIZES],
    'DP_das_Médias_Observado': [np.std(resultados_tcl[n], ddof=1) for n in SAMPLE_SIZES],
    'DP_das_Médias_Teórico': [pop_std / np.sqrt(n) for n in SAMPLE_SIZES],
    'Assimetria_das_Médias': [stats.skew(resultados_tcl[n]) for n in SAMPLE_SIZES]
})
salvar_dados(tcl_results, '01_tcl_resultados', metodo='tcl', tipo='csv')

print("\n✅ PARTE 1 CONCLUÍDA!")

# 🎯 PARTE 2: Cobertura do Intervalo de Confiança

## Objetivo
Verificar se a taxa de cobertura do IC mantém o nível de confiança nominal (95%), mesmo com dados não-normais.

## Setup
- **Distribuição**: Exponencial (mesma da Parte 1)
- **Tamanho amostral**: n = 50 (grande, atende TCL)
- **Nível de confiança**: 95%
- **Número de ICs simulados**: k = 10.000

## Pergunta Chave
Quantos dos 10.000 ICs realmente contêm a verdadeira média μ = 5?
- Esperado: ~9.500 ICs (95%)
- Se empírico ≈ teórico → IC é robusto para dados não-normais

---

In [ ]:
# PARTE 2: COBERTURA DO INTERVALO DE CONFIANÇA
print("\n" + "="*70)
print("🎯 PARTE 2: COBERTURA DO INTERVALO DE CONFIANÇA")
print("="*70)

# Parâmetros da Parte 2
CONFIDENCE_LEVEL = 0.95
ALPHA = 1 - CONFIDENCE_LEVEL
SAMPLE_SIZE_IC = 50
NUM_IC_SIMULATIONS = 10000

PARAMETROS['parte_2'] = {
    'distribuicao': 'Exponencial (mesma da Parte 1)',
    'tamanho_amostral': SAMPLE_SIZE_IC,
    'nivel_confianca': CONFIDENCE_LEVEL,
    'num_ics_simulados': NUM_IC_SIMULATIONS,
    'metodo': 'Distribuição t-student'
}

print(f"\n📊 CONFIGURAÇÃO:")
print(f"   Tamanho amostral: n = {SAMPLE_SIZE_IC}")
print(f"   Nível de confiança: {CONFIDENCE_LEVEL*100:.0f}%")
print(f"   Número de ICs: {NUM_IC_SIMULATIONS:,}")
print(f"   Método: t-student com {SAMPLE_SIZE_IC-1} graus de liberdade")

# Simular ICs
ics = []
ic_contains_mean = 0

for _ in range(NUM_IC_SIMULATIONS):
    sample = np.random.choice(population, size=SAMPLE_SIZE_IC, replace=True)
    
    # Calcular IC usando distribuição t
    sample_mean = np.mean(sample)
    sample_std = np.std(sample, ddof=1)
    standard_error = sample_std / np.sqrt(SAMPLE_SIZE_IC)
    degrees_freedom = SAMPLE_SIZE_IC - 1
    
    # Valor crítico t
    t_critical = stats.t.ppf(1 - ALPHA/2, df=degrees_freedom)
    margin_error = t_critical * standard_error
    
    ic_lower = sample_mean - margin_error
    ic_upper = sample_mean + margin_error
    ics.append((ic_lower, ic_upper))
    
    # Verificar se contém a verdadeira média
    if ic_lower <= TRUE_MEAN <= ic_upper:
        ic_contains_mean += 1

# Cálcular taxa de cobertura
coverage_rate = ic_contains_mean / NUM_IC_SIMULATIONS

print(f"\n✅ RESULTADOS:")
print(f"   ICs que contêm μ = {TRUE_MEAN}: {ic_contains_mean}/{NUM_IC_SIMULATIONS}")
print(f"   Taxa de cobertura: {coverage_rate*100:.2f}%")
print(f"   Taxa esperada: ~{CONFIDENCE_LEVEL*100:.0f}%")
print(f"   Erro: {abs(coverage_rate - CONFIDENCE_LEVEL)*100:.2f}%")

# Visualizar 100 primeiros ICs
num_vis = 100
fig, ax = plt.subplots(figsize=(12, 8))

for i in range(num_vis):
    ic_lower, ic_upper = ics[i]
    contains = (ic_lower <= TRUE_MEAN <= ic_upper)
    cor = 'green' if contains else 'red'
    
    ax.plot([ic_lower, ic_upper], [i, i], color=cor, alpha=0.6, linewidth=1)
    ax.plot(np.mean([ic_lower, ic_upper]), i, 'o', color=cor, markersize=3)

ax.axvline(TRUE_MEAN, color='blue', linestyle='--', linewidth=2, label=f'μ = {TRUE_MEAN}')
ax.set_xlabel('Valor')
ax.set_ylabel(f'IC (primeiros {num_vis})')
ax.set_title(f'Intervalos de Confiança {CONFIDENCE_LEVEL*100:.0f}% (n={SAMPLE_SIZE_IC})')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Salvar figura (em pasta IC)
salvar_figura(fig, '02_cobertura_ic', metodo='ic')

# Salvar resultados dos ICs (em pasta IC)
ic_df = pd.DataFrame({
    'IC_Inferior': [ic[0] for ic in ics],
    'IC_Superior': [ic[1] for ic in ics],
    'Contém_μ': [(ic[0] <= TRUE_MEAN <= ic[1]) for ic in ics]
})
salvar_dados(ic_df, '02_intervalos_confianca', metodo='ic', tipo='csv')

# Resumo da Parte 2
parte2_summary = {
    'Total_ICs': NUM_IC_SIMULATIONS,
    'ICs_com_media': ic_contains_mean,
    'Taxa_Cobertura_Observada': round(coverage_rate, 4),
    'Taxa_Cobertura_Esperada': CONFIDENCE_LEVEL,
    'Conclusao': f'Taxa de cobertura ({coverage_rate*100:.2f}%) ≈ esperado ({CONFIDENCE_LEVEL*100:.0f}%). IC é robusto!'
}
salvar_dados(parte2_summary, '02_ic_resumo', metodo='ic', tipo='json')

print("\n✅ PARTE 2 CONCLUÍDA!")

# 📊 PARTE 3: ANOVA One-Way com Teste de Tukey HSD

## Objetivo
Testar se há diferenças significativas entre grupos usando ANOVA e identificar quais grupos diferem.

## Cenário
Comparação do tempo de secagem de 3 revestimentos (A, B, C):
- **Revestimento A**: Média = 20 min (σ = 1.5)
- **Revestimento B**: Média = 21 min (σ = 1.5) [Similar a A]
- **Revestimento C**: Média = 28 min (σ = 1.5) [Diferente de A e B]
- **Repetições**: n = 10 por grupo

## Hipóteses
- **H₀**: μₐ = μᵦ = μ꜀ (não há diferenças)
- **H₁**: Pelo menos uma média é diferente

## Teste Post-Hoc
- **Tukey HSD** (Honestly Significant Difference) ajusta para comparações múltiplas

---

In [ ]:
# PARTE 3: ANOVA ONE-WAY COM TUKEY HSD
print("\n" + "="*70)
print("📊 PARTE 3: ANOVA ONE-WAY COM TESTE DE TUKEY HSD")
print("="*70)

# Parâmetros da Parte 3
np.random.seed(123)
N_REP = 10
MEANS = {'A': 20, 'B': 21, 'C': 28}
SIGMA = 1.5

PARAMETROS['parte_3'] = {
    'delineamento': 'ANOVA One-Way',
    'grupos': list(MEANS.keys()),
    'medias_grupo': MEANS,
    'desvio_padrao': SIGMA,
    'repeticoes_por_grupo': N_REP,
    'teste_post_hoc': 'Tukey HSD',
    'nivel_significancia': 0.05
}

# Gerar dados
grupos_labels = []
tempo_secagem = []

for grupo, media in MEANS.items():
    dados_grupo = np.random.normal(loc=media, scale=SIGMA, size=N_REP)
    grupos_labels.extend([grupo] * N_REP)
    tempo_secagem.extend(dados_grupo)

# Criar DataFrame
dados_anova = pd.DataFrame({
    'Revestimento': grupos_labels,
    'Tempo_Secagem': tempo_secagem
})

print(f"\n📊 DADOS SIMULADOS:")
print(f"   Grupos: {list(MEANS.keys())}")
print(f"   Repetições por grupo: {N_REP}")
print(f"   Total de observações: {len(tempo_secagem)}")

# Estatísticas descritivas por grupo
print(f"\n📈 ESTATÍSTICAS POR GRUPO:")
stats_by_group = dados_anova.groupby('Revestimento')['Tempo_Secagem'].agg(['count', 'mean', 'std', 'min', 'max'])
print(stats_by_group)

# ANOVA
from scipy.stats import f_oneway

grupos_dados = [dados_anova[dados_anova['Revestimento'] == g]['Tempo_Secagem'].values 
                for g in sorted(dados_anova['Revestimento'].unique())]

f_statistic, p_value = f_oneway(*grupos_dados)

print(f"\n🔍 RESULTADO DA ANOVA:")
print(f"   F-statistic: {f_statistic:.4f}")
print(f"   p-value: {p_value:.6f}")

if p_value < 0.05:
    print(f"   ✅ Significativo (p < 0.05): Há diferenças entre os grupos")
else:
    print(f"   ❌ Não significativo (p ≥ 0.05): Sem diferenças entre os grupos")

# Teste de Tukey HSD manualmente
from scipy.stats import studentized_range

# Calcular componentes para Tukey
k = len(MEANS)  # número de grupos
n = N_REP  # tamanho de cada grupo
N = k * n  # total de observações
df_error = N - k  # graus de liberdade do erro

# Variância de erro (MSE)
ss_total = np.sum((tempo_secagem - np.mean(tempo_secagem))**2)
ss_groups = sum(N_REP * (np.mean(grupos_dados[i]) - np.mean(tempo_secagem))**2 for i in range(k))
ss_error = ss_total - ss_groups
mse = ss_error / df_error

# Valor crítico de Tukey
q_critical = studentized_range(0.05, k, df_error)
se = np.sqrt(mse / N_REP)
hsd = q_critical * se

print(f"\n🔬 TESTE POST-HOC (TUKEY HSD):")
print(f"   HSD (Honestly Significant Difference): {hsd:.4f}")
print(f"   Duas médias diferem se |μᵢ - μⱼ| > HSD")

# Comparações pairwise
comparacoes = []
grupos_unicos = sorted(dados_anova['Revestimento'].unique())

print(f"\n📊 COMPARAÇÕES PAIRWISE:")
for i, g1 in enumerate(grupos_unicos):
    for g2 in grupos_unicos[i+1:]:
        media1 = np.mean(grupos_dados[grupos_unicos.index(g1)])
        media2 = np.mean(grupos_dados[grupos_unicos.index(g2)])
        diff = abs(media1 - media2)
        significante = diff > hsd
        
        comparacoes.append({
            'Grupo_1': g1,
            'Grupo_2': g2,
            'Diferença': diff,
            'HSD': hsd,
            'Significante': significante
        })
        
        status = "✅ Significante" if significante else "❌ Não significante"
        print(f"   {g1} vs {g2}: |{diff:.4f}| > {hsd:.4f}? {status}")

# Visualizar boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
ax = axes[0]
grupos_unicos = sorted(dados_anova['Revestimento'].unique())
dados_para_boxplot = [dados_anova[dados_anova['Revestimento'] == g]['Tempo_Secagem'].values 
                      for g in grupos_unicos]
bp = ax.boxplot(dados_para_boxplot, labels=grupos_unicos, patch_artist=True)
cores = ['lightblue', 'lightcoral', 'lightgreen']
for patch, color in zip(bp['boxes'], cores):
    patch.set_facecolor(color)
ax.set_ylabel('Tempo de Secagem (min)')
ax.set_xlabel('Revestimento')
ax.set_title('Distribuição do Tempo de Secagem por Revestimento')
ax.grid(alpha=0.3, axis='y')

# Histogramas
ax = axes[1]
for idx, (grupo, cor) in enumerate(zip(grupos_unicos, cores)):
    dados_grupo = dados_anova[dados_anova['Revestimento'] == grupo]['Tempo_Secagem']
    ax.hist(dados_grupo, bins=8, alpha=0.6, label=f'Revestimento {grupo}', color=cor, edgecolor='black')
ax.set_xlabel('Tempo de Secagem (min)')
ax.set_ylabel('Frequência')
ax.set_title('Histogramas Sobrepostos')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Salvar figura (em pasta ANOVA)
salvar_figura(fig, '04_anova_boxplot_histograma', metodo='anova')

# Salvar resultados (em pasta ANOVA)
salvar_dados(dados_anova, '04_dados_anova', metodo='anova', tipo='csv')
salvar_dados(stats_by_group.reset_index(), '04_anova_descritivo', metodo='anova', tipo='csv')

anova_summary = {
    'F_statistic': round(f_statistic, 4),
    'p_value': round(p_value, 6),
    'Significante': p_value < 0.05,
    'Conclusao': f'Há diferenças significativas entre revestimentos (F={f_statistic:.4f}, p={p_value:.6f})'
}
salvar_dados(anova_summary, '04_anova_resumo', metodo='anova', tipo='json')

comparacoes_df = pd.DataFrame(comparacoes)
salvar_dados(comparacoes_df, '04_tukey_comparacoes', metodo='anova', tipo='csv')

print("\n✅ PARTE 3 CONCLUÍDA!")

# 🎓 CONCLUSÕES FINAIS

## Resumo dos Resultados

### Parte 1: Teorema Central do Limite ✅
- A distribuição das médias amostrais converge para Normal mesmo com população exponencial assimétrica
- Assimetria reduz significativamente com aumento de n
- Para n = 50, a distribuição é aproximadamente Normal (assimetria ≈ 0)

### Parte 2: Intervalo de Confiança ✅
- Taxa de cobertura observada ≈ taxa esperada (95%)
- IC baseado em t-student é **robusto** para dados não-normais
- Mesmo sem distribuição normal na população, o IC funciona corretamente (graças ao TCL)

### Parte 3: ANOVA e Tukey HSD ✅
- ANOVA identificou diferenças significativas entre revestimentos
- Teste de Tukey HSD mostrou quais grupos diferem especificamente
- Método é válido mesmo com tamanhos amostrais pequenos (n=10)

## Lições Aprendidas

1. **TCL é robusto**: Funciona com qualquer distribuição (n ≥ 30)
2. **IC é confiável**: Mantém nível de confiança mesmo com dados assimétricos
3. **ANOVA é robusto**: Válido com dados não-normais e tamanhos amostrais pequenos
4. **Métodos paramétricos**: Têm fundamentação teórica sólida baseada no TCL

---

In [ ]:
# SALVAR PARÂMETROS COMPLETOS E GERAR RELATÓRIO FINAL
print("\n" + "="*70)
print("📋 SALVANDO PARÂMETROS E RELATÓRIO FINAL")
print("="*70)

# Salvar todos os parâmetros em pasta TCL (como referência geral)
salvar_dados(PARAMETROS, 'parametros_completos', metodo='tcl', tipo='json')

# Gerar relatório textual completo
relatorio_texto = f"""
================================================================================
RELATÓRIO FINAL: ANÁLISE ESTATÍSTICA COMPLETA
================================================================================

Data de Execução: {PARAMETROS['data_execucao']}
Disciplina: {PARAMETROS['disciplina']}
Referência: {PARAMETROS['referencia']}
Estrutura: {PARAMETROS['estrutura']}

================================================================================
PARTE 1: TEOREMA CENTRAL DO LIMITE (TCL)
================================================================================

Parâmetros:
- Distribuição: Exponencial
- Lambda (λ): {PARAMETROS['parte_1']['lambda']}
- Média Populacional: {PARAMETROS['parte_1']['media_populacional']}
- Tamanho da População: {PARAMETROS['parte_1']['tamanho_populacao']:,}
- Número de Simulações: {PARAMETROS['parte_1']['num_simulacoes']:,}
- Tamanhos Amostrais: {PARAMETROS['parte_1']['tamanhos_amostrais']}

Resultados:
- A assimetria da população é forte (skew ≈ {pop_skewness:.4f})
- Para n = 5: assimetria das médias ≈ {stats.skew(resultados_tcl[5]):.4f}
- Para n = 50: assimetria das médias ≈ {stats.skew(resultados_tcl[50]):.4f}
- Conclusão: TCL funciona mesmo com população assimétrica

Conclusão: ✅ VALIDADO - As médias convergem para distribuição Normal

Localização dos Resultados:
- Dados: {METODOS['tcl']['dados']}
- Gráficos: {METODOS['tcl']['imagens']}

================================================================================
PARTE 2: COBERTURA DO INTERVALO DE CONFIANÇA
================================================================================

Parâmetros:
- Distribuição: Exponencial (Parte 1)
- Tamanho Amostral: {PARAMETROS['parte_2']['tamanho_amostral']}
- Nível de Confiança: {PARAMETROS['parte_2']['nivel_confianca']*100:.0f}%
- Número de ICs Simulados: {PARAMETROS['parte_2']['num_ics_simulados']:,}
- Método: {PARAMETROS['parte_2']['metodo']}

Resultados:
- Total de ICs: {NUM_IC_SIMULATIONS:,}
- ICs que contêm μ = {TRUE_MEAN}: {ic_contains_mean:,}
- Taxa de Cobertura Observada: {coverage_rate*100:.2f}%
- Taxa de Cobertura Esperada: {CONFIDENCE_LEVEL*100:.0f}%
- Erro: {abs(coverage_rate - CONFIDENCE_LEVEL)*100:.2f}%

Conclusão: ✅ ROBUSTO - IC mantém nível nominal mesmo com dados não-normais

Localização dos Resultados:
- Dados: {METODOS['ic']['dados']}
- Gráficos: {METODOS['ic']['imagens']}

================================================================================
PARTE 3: ANOVA ONE-WAY COM TESTE DE TUKEY HSD
================================================================================

Parâmetros:
- Delineamento: {PARAMETROS['parte_3']['delineamento']}
- Grupos: {', '.join(PARAMETROS['parte_3']['grupos'])}
- Repetições por Grupo: {PARAMETROS['parte_3']['repeticoes_por_grupo']}
- Teste Post-Hoc: {PARAMETROS['parte_3']['teste_post_hoc']}
- Nível de Significância: α = {PARAMETROS['parte_3']['nivel_significancia']}

Resultados:
- F-statistic: {f_statistic:.4f}
- p-value: {p_value:.6f}
- Decisão: {"Rejeitar H₀ (há diferenças)" if p_value < 0.05 else "Não rejeitar H₀"}
- HSD (Tukey): {hsd:.4f}

Comparações:
{chr(10).join([f"  - {row['Grupo_1']} vs {row['Grupo_2']}: Diferença = {row['Diferença']:.4f} {'(SIGNIFICANTE)' if row['Significante'] else '(não significante)'}" for _, row in comparacoes_df.iterrows()])}

Conclusão: ✅ ROBUSTO - ANOVA e Tukey HSD trabalham bem com tamanhos pequenos

Localização dos Resultados:
- Dados: {METODOS['anova']['dados']}
- Gráficos: {METODOS['anova']['imagens']}

================================================================================
ESTRUTURA DE DIRETÓRIOS
================================================================================

relatorio/resultados/
├── dados/
│   ├── tcl/                    ← Resultados do TCL
│   ├── ic/                     ← Resultados do IC
│   └── anova/                  ← Resultados da ANOVA
└── imagens/
    ├── tcl/                    ← Gráficos do TCL
    ├── ic/                     ← Gráficos do IC
    └── anova/                  ← Gráficos da ANOVA

================================================================================
RECOMENDAÇÕES E INSIGHTS
================================================================================

1. O Teorema Central do Limite é uma ferramenta poderosa:
   - Justifica o uso de métodos paramétricos mesmo com dados não-normais
   - Requer n ≥ 30 em geral, mas pode variar com o grau de assimetria

2. Intervalos de Confiança são confiáveis:
   - Mantêm o nível de confiança nominal mesmo com não-normalidade
   - Use t-student em vez de z-score (mais conservador)

3. ANOVA é robusta:
   - Funciona com dados não-normais (especialmente n ≥ 10)
   - Teste de Tukey HSD é eficaz para comparações pairwise

4. Métodos Paramétricos vs Não-Paramétricos:
   - Métodos paramétricos geralmente têm mais poder
   - Sua fundamentação teórica (TCL) os torna seguros para dados reais

================================================================================
FIM DO RELATÓRIO
================================================================================
"""

# Salvar relatório em pasta TCL
with open(os.path.join(METODOS['tcl']['dados'], 'RELATORIO_FINAL.txt'), 'w', encoding='utf-8') as f:
    f.write(relatorio_texto)

print("✅ Relatório gerado e salvo!")
print(f"\n📁 Localização dos resultados:")
print(f"   TCL:")
print(f"      Dados: {METODOS['tcl']['dados']}")
print(f"      Gráficos: {METODOS['tcl']['imagens']}")
print(f"   IC:")
print(f"      Dados: {METODOS['ic']['dados']}")
print(f"      Gráficos: {METODOS['ic']['imagens']}")
print(f"   ANOVA:")
print(f"      Dados: {METODOS['anova']['dados']}")
print(f"      Gráficos: {METODOS['anova']['imagens']}")

print("\n" + "="*70)
print("✅ ANÁLISE COMPLETA FINALIZADA COM SUCESSO!")
print("="*70)

In [ ]:
# ATUALIZAR TÍTULO DO NOTEBOOK
# (Este código apenas documenta - o título já foi atualizado acima)
print("\n📌 NOTEBOOK: Análise TCL, IC e ANOVA com Resultados")
print("   Baseado em: Estudos Dirigidos 1 e 2")
print("   Foco: Simulações estatísticas e validação de métodos")